<table>
  <tr>
    <td><img src="common/RVSS-logo.png" width="400"></td>
    <td><div align="left"><font size="30">Image Processing</font></div></td>
  </tr>
</table>

In [ ]:
%matplotlib notebook

import numpy as np
import matplotlib.pyplot as plt
import math
from machinevisiontoolbox import Image, plot_point
import ipywidgets as widgets


# Images and pixels
We will start by loading an image

<p style="border:3px; border-style:solid; border-color:#FF0000; padding: 1em;">Note that in this section we will consider grey scale or monochrome images.  Have a look at the color notebook in this folder.</p>

In [ ]:
image = Image('monalisa.png', grey=True)

`image` is an object that encapsulates a a NumPy array (a python style matrix) with dimensions

In [ ]:
image.shape

which we see has 700 rows and 677 columns.

The data itself, the "internal" NumPy array can be accessed by

In [ ]:
image.image

is simply a big table of 8-bit integers which represent brightness of each pixel as a number between 0 (black) and 1 (white).

We can display it as an image

In [ ]:
image.disp();

The notebook image view is interactive. If you drift your cursor over the image it displays, beneath, the pixel coordinate and the grey value of the pixel.  

You can turn that feature off by clicking the blue button containin the "power switch" icon.

# Spatial operators
## Image smoothing by averaging

The image of the Mona Lisa looks rather grainy, the paint is cracked, but it is over 500 years old.

We could smooth it out by local averaging, where every pixel in the output image is the average of all pixels in a $5 \times 5$ window about the corresponding input pixel.  We first create a kernel

In [ ]:
kernel = np.ones((5,5),np.float32) / 25
kernel

where the sum of all values is equal to one.  For a given input pixel, say (20,30), the $5 \times 5$ window is

In [ ]:
window = image.image[30-2:30+3,20-2:20+3]

<p style="border:3px; border-style:solid; border-color:#FF0000; padding: 1em;">Note that we swapped the 20 and 30.  (20,30) means the horizontal coordinate is 20 and the vertical coordinate is 30, but when we index into a 2D array the first index is row (vertical direction) and the second index is column (horizontal direction).</p>
<p style="border:3px; border-style:solid; border-color:#FF0000; padding: 1em;">Note that the range 30-2:30+3 includes 30-2, 30-1, 30-0, 30+1, 30+2.  In python the end value of  a range is not included in the set.</p>

In [ ]:
z = window * kernel
z

In [ ]:
np.sum(z)

and this is the value of pixel (20,30) in the output image.  

We need do this for every pixel in the image, and we could use a pair of nested `for` loops but that's not very efficient.  We can use optimised code in OpenCV

In [ ]:
smoothed = image.convolve(kernel)
smoothed.disp();

### Exercise

Vary the dimensions of the kernel to see what effect it has. 

## Gaussian blur

For image smoothing it is best to use a kernel that is isotropic and symmetric such as a 2D Gaussian
$$G(u,v) = \frac{1}{2\pi\sigma^2}e^{-\frac{u^2+v^2}{2\sigma^2}}$$

In [ ]:
w = 5
k = range(-w, w+1)
sigma = 2
[U,V] = np.meshgrid(k, k)
kernel = 1/(2*math.pi*sigma**2)*np.exp(-(U**2+V**2)/(2*sigma**2))
kernel
from mpl_toolkits.mplot3d import axes3d
from matplotlib import cm
fig = plt.figure()
ax = fig.gca(projection='3d')
surf = ax.plot_surface(U, V, kernel, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)
# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.set_xlabel('U')
ax.set_ylabel('V')
ax.set_zlabel('K(U,V)')

We can blur our image with this kernel

In [ ]:
smoothed = image.convolve(kernel)
smoothed.disp();

We can do this in a single step where we pass in the image

In [ ]:
blur = image.smooth(3, 5)
blur.disp()

### Exercise

* Vary the dimensions of the kernel to see what effect it has. 
* Vary the standard deviation

## Finding edges
We can use 2D filtering to find edges as well.  This convolution kernel will find vertical edges.  The intuition is that each row of this kernel subtracts the pixel to the left from the pixel to the right, which will give a positive value if the intensity is increasing left to right.
<p style="border:3px; border-style:solid; border-color:#FF0000; padding: 1em;">You may often see this filter kernel written with the first and third columns swapped.  This is appropriate if you perform correlation, not convolution. These are two similar operations but differ in the kernel being reflected about its centre.</p>



In [ ]:
kernel = np.array( [ [1, 0, -1],
                     [2, 0, -2],
                     [1, 0, -1] ]) / 8

In [ ]:
penguins = Image('penguins.png', grey=True)
penguins.disp();

In [ ]:
edgy = penguins.convolve(kernel)
edgy.disp(colormap='signed');                

The image is displayed with a color map that shows negative numbers as red and positive numbers as blue.  Zoom in on the outline of the "P" (use the second button from the right in the bottom toolbar) and you can see that the intensity goes up (blue) on the left side of the "P", from the grey background to the white paint. It goes down (red) on right of the stem, from the white paint to the gray background.

<p style="border:3px; border-style:solid; border-color:#FF0000; padding: 1em;">Note that we tell filter2D to output a signed floating point image.  The result, at each pixel, can be positive or negative.  By default the output image will be the same as the input image (signalled by the -1 second argument used in previous examples) but that cannot represent negative numbers.  In the land of 8-bit unsigned numbers 6-4=2 but 4-6=0.</p>

We can find the horizontal edges using the transpose of the kernel

In [ ]:
edgy = penguins.convolve(kernel.T)
edgy.disp(colormap='signed');    

# Harris corner features
## From first principles

We now have all the ingredients (and knowledge) needed to find interest points.

In [ ]:
image = Image('castle.png', grey=True)

The fundamental intuition behind interest points is that they have a high gradient in two orthogonal directions, but not necessarily the u- and v-axis directions.

However we start by computing the gradient in the u- and v-axis directions and then for every pixel compute this matrix
$$\mathbf{A} = \begin{pmatrix} \mathbf{G}_\sigma * \mathbf{I}_u^2  & \mathbf{G}_\sigma * \mathbf{I}_u \mathbf{I}_v \\ \mathbf{G}_\sigma * \mathbf{I}_u \mathbf{I}_v & \mathbf{G}_\sigma * \mathbf{I}_v^2 \end{pmatrix}$$
This $2 \times 2$ symmetric matrix is called the structure tensor.
It captures the intensity structure of the local neighborhood and its eigenvalues provide a rotationally invariant description of the neighborhood. The elements of the $\mathbf{A}$ matrix are computed from the image gradients, squared or multiplied, and then smoothed using a weighting matrix. The latter reduces noise and improves the stability and reliability of the detector.

In [ ]:
# compute derivatives

kernel = np.array( [ [1, 0, -1],
                     [2, 0, -2],
                     [1, 0, -1] ]) / 8

Iu = image.convolve(kernel)
Iv = image.convolve(kernel.T)

# make a Gaussian smoothing kernel 11x11 with sigma=1
w2 = 5  # half width
k = range(-w2, w2+1)
sigma = 1
[U,V] = np.meshgrid(k, k)
kgaussian = 1.0 / (2 * math.pi * sigma**2) * np.exp(-(U**2 + V**2) / (2 * sigma**2))

# could also use kgaussian = kgauss(sigma, w2)

# compute the 3 unique elements of the structure tensor
Ix = (Iu * Iu).convolve(kgaussian)
Iy = (Iv * Iv).convolve(kgaussian)
Ixy = (Iu * Iv).convolve(kgaussian)

An interest point $(u, v)$ is one where whatever direction we move the window it rapidly becomes dissimilar to the original region. If we consider the original image $\mathbf{I}$ as a surface the eigenvalues of $\mathbf{A}$ are the principal curvatures of the surface at that point: 

* If both eigenvalues are small then the surface is flat, that is the image region has approximately constant local intensity. 
* If one eigenvalue is high and the other low, then the surface is ridge shaped which indicates an edge. 
* If both eigenvalues are high the surface is sharply peaked which we consider to be a corner.

The well known Shi-Tomasi detector considers the strength of the corner, or cornerness, as the minimum eigenvalue
$$C_{\mbox{ST}}(u,v) = \min( \lambda_1, \lambda_2)$$
where $\lambda_i$ are the eigenvalues of $\mathbf{A}$. Points in the image for which this measure is high are referred to as ["good features to track"](http://www.ai.mit.edu/courses/6.891/handouts/shi94good.pdf). 

The Harris detector is based on this same insight but defines corner strength as $$C_{H}(u,v) = \mbox{det}(\mathbf{A}) - k \mbox{tr}(\mathbf{A})$$
and again a large value represents a strong, distinct, corner. Since $\mbox{det}(A) = \lambda_1  \lambda_2$ and
$\mbox{tr}(A) = \lambda_1 + \lambda_2$ the Harris detector responds when both eigenvalues are large and elegantly avoids computing the eigenvalues of A which has a somewhat higher computational cost.  Typically $k=0.04$.

We compute this for all pixels at once

In [ ]:
rawC = (Ix * Iy - Ixy * Ixy) - 0.04 * (Ix + Iy)**2
rawC.disp(colormap='signed');

which shows the raw value of $C_H$. If we zoom in and examine the image we will see that large areas of smooth background are around zero, edges are negative and corners of things (like the letters) have a positive value.

Next we need to find the coordinates of those positive patches values.  For every pixel we find the largest value in a $5\times 5$ window around each pixel but not including the centre pixel itself.

In [ ]:
# create the structuring element
SE = np.ones((5,5), np.uint8)
SE[2,2] = 0

# find the maximum value around each pixel
surrounding_max = rawC.dilate(SE)

We've done this using a morphological filtering operation called dilation, you can find more details from the documentation page for [`dilate`](https://docs.opencv.org/master/d4/d86/group__imgproc__filter.html#ga4ff0f3318642c4f469d0e11f242f3b6c).

Next we find all the pixels whose value is greater than those surrounding it (which we just computed). This is a common trick in computer vision when we are looking for peaks &ndash; it's called non-local maxima suppresion.

In [ ]:
idx = np.nonzero((rawC > surrounding_max).image)  # find the indices of all the peaks
peakvals = rawC.image[idx]  # find the peak values
k = np.argsort(-peakvals)  # sort those values into descending order
print('%d peaks found\n' % (len(k),))  # report how many peaks were found
k = k[:200]  # take first 200, these are the strongest peaks

# find the u, v coordinates as two separate lists
u = np.array([idx[0][i] for i in k])
v = np.array([idx[1][i] for i in k])

# display the image, and overlay with markers for each detected corner feature
image.disp();
plot_point((v,u), 'b+')

Zoom in on the letters and you will see that the features are placed on sharp corners.

### Summary

Developed by Chris Harris and Mike Stephens in 1988 the [Harris detector](https://en.wikipedia.org/wiki/Harris_Corner_Detector) was THE detector used in computer vision until
[SIFT](https://en.wikipedia.org/wiki/Scale-invariant_feature_transform) was developed by David Lowe in 1999.
SIFT is very powerful (it has the very useful property called scale-invariance which we won't have time to cover) but was [patented by the University of British Columbia](https://patents.google.com/patent/US6711293) and researchers were therefore reluctant to be too dependent on it. [SURF](https://en.wikipedia.org/wiki/Speeded_up_robust_features) was  developed by Herbert Bay, Tinne Tuytelaars, and Luc Van Gool and published in 2006. It has similar functionality to SIFT but is much faster.  [SURF is also patented](https://worldwide.espacenet.com/patent/search/family/036954920/publication/US2009238460A1?q=pn%3DUS2009238460) but nobody seems to worry too much about this one...

## Using OpenCV function Harris corner detection

The OpenCV library provides a function to do the first part of this operation, producing the raw corner strength image.

In [ ]:
harris = cv2.cornerHarris(image, 2, 3, 0.04)
idisp(harris)

# SIFT features

In [ ]:
image = Image('common/images/butterfly.jpg')

kp = image.SIFT()
print("%d features found" % len(kp))


Each feature (sometimes called keypoints) that is found has a coordinate (estimated to subpixel accuracy), the first keypoint is centered at

In [ ]:
kp[0].pt

Unlike the Harris feature, the SURF feature also has a size (in pixels)

In [ ]:
kp[0].scale

as well as a characteristic angle

In [ ]:
kp[0].orientation

and we can plot the features, their size and orientation

In [ ]:
image.disp()
kp.drawKeypoints()

There is also strength or response that indicates how unique the feature is

In [ ]:
kp[0].strength

Each feature point has a descriptor which is a summary of the pixel values within the variable size disk surrounding it. This is given by the descriptor which is a vector of 64 floats


In [ ]:
kp[0].descriptor

If we moved the camera to shrink the image by a factor of two the size of all the support region disks would reduce by a factor of two, but the descriptor would remain approximately constant.  If we rotated the camera by 90degrees the characgteristic angle of all the features would change by 90degrees but the feature size and the descriptor would remain approximately constant.

Next we will use SURF to find corresponding features in a pair of images

In [ ]:
kp1 = Image('common/images/joker1_50.jpg').SIFT()
kp2 = Image('common/images/joker2_50.jpg').SIFT()

Then we find the features and descriptors

Then we create a "matcher" object and give it the two sets of descriptors.  It finds the best matches and for each match computes a `distance` which is a measure of dissimilarity.  We sort the matches into decreasing similarity

In [ ]:
matches = kp1.match(kp2)
# Sort them in the order of their difference.
matches = sorted(matches, key = lambda x: x.distance, reverse=True)

then plot the best 50 matches.  The `drawMatches` method renders the matches onto an image and returns an image

In [ ]:
matches[0][0].queryIdx

In [ ]:
dir(matches[0])

In [ ]:
# Draw first 50 matches. 
img3 = cv2.drawMatches(img1,kp1, img2,kp2, matches[:50], None, matchColor=(0,0,255),flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
idisp(img3)